# DonorData basics: creating, syncing, slicing, and saving

Every other tutorial in this section starts from a `DonorData` object and does something
with it. This one doesn't do any analysis: it just covers how `DonorData` itself works,
how to build one from your own genotype and single-cell data, how the donor axis stays in
sync when you slice, and how to save/reload it.

`DonorData` holds two `AnnData` (or `MuData`) objects that share a donor axis:

- **`G`**: donor-level data, e.g. genotypes (`donors x variants`)
- **`C`**: cell-level data, e.g. expression (`cells x genes`), with each cell tagged by
  which donor it came from

## Building `G` and `C` from scratch

To keep things fast and dependency-free, we'll build tiny toy versions of `G` and `C` by
hand first. The next section shows how you'd get `G` from a real genotype file instead.

### Cell-level data (`C`)

In [1]:
import numpy as np
import pandas as pd
from anndata import AnnData

from cellink import DonorData

n_cells, n_genes = 6, 4
X = np.random.poisson(5, size=(n_cells, n_genes)).astype(float)
obs = pd.DataFrame(
    {
        "donor_id": ["D0", "D0", "D1", "D1", "D2", "D2"],
        "celltype": ["CD4", "CD8", "CD4", "CD8", "CD4", "CD8"],
    },
    index=[f"cell{i}" for i in range(n_cells)],
)
var = pd.DataFrame(index=[f"GENE{i}" for i in range(n_genes)])
adata = AnnData(X=X, obs=obs, var=var)
adata

/lustre/groups/ml01/workspace/lucas.arnoldt/cellink/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AnnData object with n_obs × n_vars = 6 × 4
    obs: 'donor_id', 'celltype'

The only thing `DonorData` needs from `C` is a column that says which donor each cell
belongs to (`"donor_id"` by default, but you can point it at any column via the
`donor_id=` argument when you build `DonorData`).

### Donor-level data (`G`)

In [2]:
n_donors, n_snps = 3, 5
G_X = np.random.binomial(2, 0.3, size=(n_donors, n_snps)).astype(float)
G_obs = pd.DataFrame({"donor_id": ["D0", "D1", "D2"]})
G_var = pd.DataFrame(
    {"chrom": "1", "pos": np.arange(1, n_snps + 1)},
    index=[f"SNP{i}" for i in range(n_snps)],
)
gdata = AnnData(X=G_X, obs=G_obs, var=G_var)
gdata.obs

/lustre/groups/ml01/workspace/lucas.arnoldt/cellink/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


,donor_id
0,D0
1,D1
2,D2


`G` identifies donors the same way, but more flexibly: either `G.obs_names` *is* the donor
id already, or, like here, it's a regular column in `G.obs`. `DonorData` will set it as
the index for you either way, so both forms work without any extra prep.

### Creating the `DonorData` object

In [3]:
dd = DonorData(G=gdata, C=adata)
dd

╔═ DonorData(n_donors=3, n_cells_per_donor=[2-2], donor_id='donor_id') ═════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃ G (donors)                                  ┃ C (cells)                                   ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ AnnData object with n_obs × n_vars = 3 × 5  │ AnnData object with n_obs × n_vars = 6 × 4  │ ║
║ │     var: 'chrom', 'pos'                     │     obs: 'donor_id', 'celltype'             │ ║
║ └─────────────────────────────────────────────┴─────────────────────────────────────────────┘ ║
╚═══════════════════════════════════════════════════════════════════════════════════════════════╝

In [4]:
# gdata's donor_id column became its index during construction
dd.G.obs.index.name

'donor_id'

`G`/`C` can also be `MuData` objects (e.g. a multiome `C` with an RNA and an ATAC
modality). `DonorData` doesn't care, as long as the donor column/index is there.

### Keeping donors in sync

Building a `DonorData` only keeps donors present in **both** `G` and `C`, and reorders
`C`'s rows to match `G`'s donor order. A donor with genotypes but no cells (or the other
way around) is silently dropped rather than kept as a half-empty entry. Note that this
happens in place on the `G`/`C` objects you passed in, so pass `.copy()`s if you need the
originals untouched afterwards.

In [5]:
# add a 4th donor to G that has no cells in C
G_obs_extra = pd.DataFrame({"donor_id": ["D0", "D1", "D2", "D3"]})
gdata_extra = AnnData(
    X=np.random.binomial(2, 0.3, size=(4, n_snps)).astype(float),
    obs=G_obs_extra,
    var=G_var,
)
donors_in_G = list(G_obs_extra["donor_id"])  # DonorData sets this as the index in place below

dd_extra = DonorData(G=gdata_extra, C=adata)
print(f"G had donors {donors_in_G}, DonorData kept {list(dd_extra.G.obs.index)}")

G had donors ['D0', 'D1', 'D2', 'D3'], DonorData kept ['D0', 'D1', 'D2']


/lustre/groups/ml01/workspace/lucas.arnoldt/cellink/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


## Loading genotype data from real files

Building `G` by hand is only for this tutorial; normally it comes from a genotype file.
`cellink.io` reads several formats directly into the `AnnData` shape `DonorData` expects.

### From PGEN, via `stream_pgen_to_zarr` (start here)

For most workflows this is the one to reach for: a single function call directly on the
compact PLINK2 format most GWAS/biobank pipelines already produce, or can export to via
`plink2 --make-pgen`. It streams the file in fixed-size blocks so memory use stays
bounded regardless of file size, and `read_pgen_zarr` loads the result back with a
Dask-backed `X`. The `cellink-pgen` command-line tool ({doc}`../api/cli`) wraps the same
conversion for use outside Python.

It does have a few tuning knobs, but the defaults are reasonable for typical use and you
usually don't need to touch them:

- `chunk_samples=4096` / `chunk_variants=2048`: the output Zarr store's chunk shape.
  Only worth changing for an unusual access pattern (e.g. mostly per-variant lookups on
  a very large cohort) or a dataset far smaller/larger than these defaults assume.
- `memory_limit_gb=10.0`: caps how many variants get buffered per read block. Lower it
  on memory-constrained machines, raise it if you have RAM to spare and want faster
  conversion.

In [6]:
import tempfile
from pathlib import Path

from cellink.io import read_pgen_zarr, stream_pgen_to_zarr

pgen_out = Path(tempfile.mkdtemp()) / "pgen_demo.zarr"
stream_pgen_to_zarr(
    "../../tests/data/simulated_genotype_calls.pgen",
    str(pgen_out),
    # small values here just because this demo file only has 100 samples / 1000 variants;
    # leave chunk_samples/chunk_variants/memory_limit_gb at their defaults for real data
    chunk_samples=50,
    chunk_variants=200,
    memory_limit_gb=2.0,
)
gdata_pgen = read_pgen_zarr(str(pgen_out))
gdata_pgen.shape

[2026-07-11 17:36:22,566] INFO:cellink.io._pgen:   simulated_genotype_calls.pgen: 100 samples x 1,000 variants (using 1,000)


[2026-07-11 17:36:22,567] INFO:cellink.io._pgen: Total: 100 samples × 1,000 variants (dense)


[2026-07-11 17:36:22,569] INFO:cellink.io._pgen: Loading sample metadata from ../../tests/data/simulated_genotype_calls.psam ...


[2026-07-11 17:36:22,607] INFO:cellink.io._pgen: Creating dense X (100 × 1,000) chunks=(50, 200) ...


[2026-07-11 17:36:22,701] INFO:cellink.io._pgen: Reading (dense) simulated_genotype_calls.pgen ...


...

  simulated_genotype_calls: 100%|██████████| 1000/1000 [00:00<00:00, 5834.68var/s]

[2026-07-11 17:36:22,878] INFO:cellink.io._pgen: Writing obs / var / metadata ...


[2026-07-11 17:36:22,968] INFO:cellink.io._pgen: ✓ Done → /tmp/tmpwu4s7bxq/pgen_demo.zarr


/lustre/groups/ml01/workspace/lucas.arnoldt/cellink/lib/python3.12/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


(100, 1000)

### From VCF, via `vcf2zarr` (if you need more than genotypes)

A `.pgen` file only ever stores hard-call genotypes, nothing else. If you need
QUAL/FILTER/INFO fields, per-call FORMAT fields (`GQ`, `DP`, `AD`, phasing, ...), or your
data only exists as VCF in the first place, `read_sgkit_zarr` reads the
[sgkit](https://sgkit-dev.github.io/sgkit/) Zarr store produced by
[`vcf2zarr`](https://github.com/sgkit-dev/bio2zarr), which converts VCF files in two
steps:

- `vcf2zarr explode` reads the VCF(s) once into an intermediate columnar format (ICF).
- `vcf2zarr encode` turns the ICF into the actual Zarr store; this is where the chunk
  sizes you'll want to think about are set.

Reasonable starting points: chunk sizes in the tens of thousands for variants, and large
enough for samples that your whole cohort fits in a single chunk for datasets up to a
few thousand donors. Run `encode` single-process for modest VCFs (the default); for
whole-chromosome or biobank-scale conversions, use multiple worker processes together
with a memory cap (e.g. `32G`) so peak memory stays bounded.

In [7]:
from cellink.io import read_sgkit_zarr

# read_plink() and read_bgen() work the same way for PLINK1 and BGEN files
gdata_real = read_sgkit_zarr("../../tests/data/simulated_genotype_calls.vcz")
gdata_real.shape

(1000, 1999)

Because it goes through the full VCF, everything the VCF carries beyond genotypes comes
along too: QUAL/FILTER/INFO fields end up as extra `var` columns, and any per-call
FORMAT field (`GQ`, `DP`, `AD`, phasing, ...) becomes a `layer` matching `X`'s shape:

In [8]:
gdata_real.var.columns

Index(['PR', 'contig', 'id', 'id_mask', 'pos', 'quality', 'a0', 'a1', 'chrom'], dtype='object')

In [9]:
gdata_real.layers.keys()

KeysView(Layers with keys: GENOTYPE_PHASED, PHASE_0)

## Simple operations

### Selecting subsets

`DonorData` indexes along all four axes at once: `dd[G_obs, G_var, C_obs, C_var]`. Use
`...` to leave the rest untouched, and notice that selecting donors from `G` also
filters `C` down to their cells, and vice versa.

In [10]:
dd[["D0", "D1"]]

╔═ DonorData(n_donors=2, n_cells_per_donor=[2-2], donor_id='donor_id') ═══════════════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃ G (donors)                                         ┃ C (cells)                                          ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ View of AnnData object with n_obs × n_vars = 2 × 5 │ View of AnnData object with n_obs × n_vars = 4 × 4 │ ║
║ │     var: 'chrom', 'pos'                            │     obs: 'donor_id', 'celltype'                    │ ║
║ └────────────────────────────────────────────────────┴────────────────────────────────────────────────────┘ ║
╚═════════════════════════════════════════════════════════════════════════════════════════════════════════════╝

In [11]:
dd[..., dd.C.obs["celltype"] == "CD4", :]

╔═ DonorData(n_donors=3, n_cells_per_donor=[1-1], donor_id='donor_id') ═══════════════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃ G (donors)                                         ┃ C (cells)                                          ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ View of AnnData object with n_obs × n_vars = 3 × 5 │ View of AnnData object with n_obs × n_vars = 3 × 4 │ ║
║ │     var: 'chrom', 'pos'                            │     obs: 'donor_id', 'celltype'                    │ ║
║ └────────────────────────────────────────────────────┴────────────────────────────────────────────────────┘ ║
╚═════════════════════════════════════════════════════════════════════════════════════════════════════════════╝

`.sel()` does the same thing with keyword arguments (`G_obs`, `G_var`, `C_obs`, `C_var`),
which reads a little clearer once more than one axis is involved.

In [12]:
dd.sel(C_obs=(dd.C.obs["celltype"] == "CD4").values)

╔═ DonorData(n_donors=3, n_cells_per_donor=[1-1], donor_id='donor_id') ═══════════════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃ G (donors)                                         ┃ C (cells)                                          ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ View of AnnData object with n_obs × n_vars = 3 × 5 │ View of AnnData object with n_obs × n_vars = 3 × 4 │ ║
║ │     var: 'chrom', 'pos'                            │     obs: 'donor_id', 'celltype'                    │ ║
║ └────────────────────────────────────────────────────┴────────────────────────────────────────────────────┘ ║
╚═════════════════════════════════════════════════════════════════════════════════════════════════════════════╝

### Aggregating cells to donor level

`.aggregate()` pseudobulks `C` down to one row per donor and stores the result in
`G.obsm` (or `G.obs`, via `add_to_obs=True`, for scalar covariates).

In [13]:
dd.aggregate(key_added="mean_expr")
dd.G.obsm["mean_expr"]

,GENE0,GENE1,GENE2,GENE3
donor_id,,,,
D0,3.0,4.5,3.5,4.0
D1,6.5,6.0,3.5,2.5
D2,6.0,6.5,6.5,3.0


## Saving and loading

`write_h5_dd`/`write_zarr_dd` save `G` and `C` together in a single file;
`read_h5_dd`/`read_zarr_dd` load them straight back into a `DonorData`.

In [14]:
import tempfile
from pathlib import Path

from cellink.io import read_h5_dd

tmp_dir = Path(tempfile.mkdtemp())
dd.write_h5_dd(str(tmp_dir / "dd_basics.dd.h5"))

dd_loaded = read_h5_dd(str(tmp_dir / "dd_basics.dd.h5"))
dd_loaded

╔═ DonorData(n_donors=3, n_cells_per_donor=[2-2], donor_id='donor_id') ═════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃ G (donors)                                  ┃ C (cells)                                   ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ AnnData object with n_obs × n_vars = 3 × 5  │ AnnData object with n_obs × n_vars = 6 × 4  │ ║
║ │     var: 'chrom', 'pos'                     │     obs: 'donor_id', 'celltype'             │ ║
║ │     obsm: 'mean_expr'                       │                                             │ ║
║ └─────────────────────────────────────────────┴─────────────────────────────────────────────┘ ║
╚═══════════════════════════════════════════════════════════════════════════════════════════════╝

## Next steps

With your own `DonorData` in hand, {doc}`explore_annotations` covers variant QC and
annotation, and {doc}`pseudobulk_eqtl` walks through a first real analysis.